# Notebook 02: Mô Hình Dự Đoán + Hiệu Chỉnh Xác Suất

**PHIÊN BẢN ĐÃ SỬA - Điều Chỉnh CV + Không Rò Rỉ Dữ Liệu**

**Các thay đổi chính:**
- Điều chỉnh siêu tham số bằng 5-fold CV trên tập huấn luyện (3,000 mẫu)
- Hiệu chỉnh trên TOÀN BỘ tập kiểm định (1,000 mẫu - KHÔNG dùng cho điều chỉnh)
- So sánh: Isotonic Regression vs. Platt Scaling
- Chỉ dùng đặc trưng pre-treatment (không rò rỉ dữ liệu)

**Ngày:** 06/11/2026

**Tác giả:** Phan Hồ Ngọc Hân

In [1]:
# ============================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ============================================
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Các mô hình từ Sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

# Các tiện ích từ Sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    classification_report,
    confusion_matrix
)

# Trực quan hóa dữ liệu
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập seed cố định để đảm bảo khả năng tái lập kết quả
SEED = 42
np.random.seed(SEED)

# Thiết lập hiển thị dữ liệu
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"Đã import hoàn tất - SEED = {SEED}")

Đã import hoàn tất - SEED = 42


## 1. Tải Dữ Liệu Đã Xử Lý

In [2]:
# ============================================
# TẢI DỮ LIỆU ĐÃ XỬ LÝ
# ============================================
data_path = Path("../data/processed/marketing_campaign_processed.parquet")
metadata_path = Path("../data/processed/feature_metadata.json")

# Kiểm tra sự tồn tại của file dữ liệu
if not data_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file dữ liệu: {data_path}\n"
        "Vui lòng chạy Notebook 01 trước!"
    )

# Tải dữ liệu từ file parquet
df = pd.read_parquet(data_path)

# Tải metadata từ file json
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"Đã tải dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Đã tải metadata: {len(metadata)} mục")

# Trích xuất danh sách đặc trưng từ metadata
MODELING_FEATURES = metadata['modeling_features']

print(f"\nCác đặc trưng dùng cho mô hình (chỉ pre-treatment): {len(MODELING_FEATURES)}")
for i, feat in enumerate(MODELING_FEATURES, 1):
    print(f"  {i}. {feat}")

Đã tải dữ liệu: 5,000 dòng × 20 cột
Đã tải metadata: 9 mục

Các đặc trưng dùng cho mô hình (chỉ pre-treatment): 8
  1. campaign_id
  2. channel
  3. device
  4. country
  5. segment
  6. prior_visits_30d
  7. prior_spend_180d
  8. value_tier


## 2. Phân Chia Dữ Liệu Theo Thời Gian

In [3]:
# ============================================
# PHÂN CHIA DỮ LIỆU THEO CÁC VÙNG THỜI GIAN
# ============================================

# Phân chia theo cột split (đã tạo trong Notebook 01)
df_train = df[df['split'] == 'train'].copy()
df_val = df[df['split'] == 'validation'].copy()
df_test = df[df['split'] == 'locked_test'].copy()
df_oot = df[df['split'] == 'out_of_time'].copy()

print("=" * 70)
print("PHÂN CHIA THEO THỜI GIAN")
print("=" * 70)
print(f"Huấn luyện:  {len(df_train):,} mẫu (60%)")
print(f"Kiểm định:   {len(df_val):,} mẫu (20%) - CHỈ dùng cho hiệu chỉnh, KHÔNG dùng cho điều chỉnh")
print(f"Kiểm tra:    {len(df_test):,} mẫu (10%) - Khóa, chỉ mở một lần vào cuối")
print(f"Out-of-Time: {len(df_oot):,} mẫu (10%) - Phát hiện drift")
print("=" * 70)

PHÂN CHIA THEO THỜI GIAN
Huấn luyện:  3,000 mẫu (60%)
Kiểm định:   1,000 mẫu (20%) - CHỈ dùng cho hiệu chỉnh, KHÔNG dùng cho điều chỉnh
Kiểm tra:    500 mẫu (10%) - Khóa, chỉ mở một lần vào cuối
Out-of-Time: 500 mẫu (10%) - Phát hiện drift


In [4]:
# ============================================
# CHUẨN BỊ X VÀ y
# ============================================

# Đặc trưng (CHỈ PRE-TREATMENT)
X_train = df_train[MODELING_FEATURES].copy()
X_val = df_val[MODELING_FEATURES].copy()
X_test = df_test[MODELING_FEATURES].copy()
X_oot = df_oot[MODELING_FEATURES].copy()

# Biến mục tiêu (conversion - tỷ lệ chuyển đổi)
y_train = df_train['conversion'].values
y_val = df_val['conversion'].values
y_test = df_test['conversion'].values
y_oot = df_oot['conversion'].values

print("\nĐã chuẩn bị X và y:")
print(f"   X_train: {X_train.shape}")
print(f"   X_val:   {X_val.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   X_oot:   {X_oot.shape}")

print(f"\nTỷ lệ chuyển đổi:")
print(f"   Huấn luyện:  {y_train.mean()*100:.2f}%")
print(f"   Kiểm định:   {y_val.mean()*100:.2f}%")
print(f"   Kiểm tra:    {y_test.mean()*100:.2f}%")
print(f"   Out-of-Time: {y_oot.mean()*100:.2f}%")


Đã chuẩn bị X và y:
   X_train: (3000, 8)
   X_val:   (1000, 8)
   X_test:  (500, 8)
   X_oot:   (500, 8)

Tỷ lệ chuyển đổi:
   Huấn luyện:  10.00%
   Kiểm định:   10.80%
   Kiểm tra:    9.60%
   Out-of-Time: 10.20%


## 3. Điều Chỉnh Siêu Tham Số với GridSearchCV

### QUAN TRỌNG: Điều Chỉnh CV Trên Tập Huấn Luyện

**Cách tiếp cận cũ (SAI):**
- Điều chỉnh trên một phần tập kiểm định
- Hiệu chỉnh trên phần kiểm định còn lại
- **Vấn đề:** Mỗi phần quá nhỏ (500 mẫu)

**Cách tiếp cận mới (ĐÚNG):**
- **Điều chỉnh:** 5-fold CV trên tập huấn luyện (3,000 mẫu)
- **Hiệu chỉnh:** TOÀN BỘ tập kiểm định (1,000 mẫu)
- **Đánh giá:** Tập kiểm tra đã khóa (500 mẫu)

→ Tập kiểm định KHÔNG BAO GIỜ được dùng trong giai đoạn điều chỉnh!

In [5]:
# ============================================
# XÁC ĐỊNH CÁC ĐẶC TRƯNG PHÂN LOẠI
# ============================================

# Danh sách các cột phân loại (categorical)
categorical_features = ['campaign_id', 'channel', 'device', 'country', 'segment', 'value_tier']

# Lấy chỉ số của các đặc trưng phân loại trong MODELING_FEATURES
cat_indices = [i for i, feat in enumerate(MODELING_FEATURES) if feat in categorical_features]

print(f"Các đặc trưng phân loại: {len(cat_indices)} đặc trưng")
print(f"   Chỉ số: {cat_indices}")
print(f"   Tên: {[MODELING_FEATURES[i] for i in cat_indices]}")

Các đặc trưng phân loại: 6 đặc trưng
   Chỉ số: [0, 1, 2, 3, 4, 7]
   Tên: ['campaign_id', 'channel', 'device', 'country', 'segment', 'value_tier']


In [6]:
# ============================================
# ĐIỀU CHỈNH SIÊU THAM SỐ VỚI 5-FOLD CV
# ============================================

print("\n" + "=" * 70)
print("ĐIỀU CHỈNH SIÊU THAM SỐ (5-fold CV trên Tập Huấn Luyện)")
print("=" * 70)

# Định nghĩa lưới tham số
param_grid = {
    'max_depth': [4, 6, 8],              # Độ sâu tối đa của cây quyết định
    'max_iter': [50, 100, 150],          # Số lần lặp tối đa
    'learning_rate': [0.05, 0.1, 0.15]   # Tốc độ học
}

print(f"\nLưới tham số: {len(param_grid['max_depth']) * len(param_grid['max_iter']) * len(param_grid['learning_rate'])} tổ hợp")
print(f"   max_depth: {param_grid['max_depth']}")
print(f"   max_iter: {param_grid['max_iter']}")
print(f"   learning_rate: {param_grid['learning_rate']}")

# Mô hình cơ sở
base_model = HistGradientBoostingClassifier(
    categorical_features=cat_indices,  # Chỉ định các đặc trưng phân loại
    random_state=SEED                  # Đảm bảo kết quả có thể tái lập
)

# GridSearchCV với 5-fold cross-validation
cv_model = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,                        # 5-fold cross-validation
    scoring='neg_brier_score',   # Tối ưu hóa theo Brier score
    n_jobs=-1,                   # Sử dụng tất cả CPU cores
    verbose=1
)

print("\nĐang điều chỉnh... (có thể mất 1-2 phút)")

# Huấn luyện trên TẬP HUẤN LUYỆN (tập kiểm định KHÔNG được sử dụng)
cv_model.fit(X_train, y_train)

print("\nĐiều chỉnh hoàn tất!")
print(f"\nTham số tốt nhất:")
for param, value in cv_model.best_params_.items():
    print(f"   {param}: {value}")

print(f"\nBrier score CV tốt nhất: {-cv_model.best_score_:.4f}")

# Trích xuất mô hình tốt nhất
best_model = cv_model.best_estimator_


ĐIỀU CHỈNH SIÊU THAM SỐ (5-fold CV trên Tập Huấn Luyện)

Lưới tham số: 27 tổ hợp
   max_depth: [4, 6, 8]
   max_iter: [50, 100, 150]
   learning_rate: [0.05, 0.1, 0.15]

Đang điều chỉnh... (có thể mất 1-2 phút)
Fitting 5 folds for each of 27 candidates, totalling 135 fits



Điều chỉnh hoàn tất!

Tham số tốt nhất:
   learning_rate: 0.1
   max_depth: 4
   max_iter: 50

Brier score CV tốt nhất: 0.0889


## 4. Dự Đoán Thô (Chưa Hiệu Chỉnh)

In [7]:
# ============================================
# DỰ ĐOÁN THÔ (TRƯỚC KHI HIỆU CHỈNH)
# ============================================

print("\n" + "=" * 70)
print("DỰ ĐOÁN THÔ (Chưa Hiệu Chỉnh)")
print("=" * 70)

# Lấy xác suất dự đoán (cột thứ 2 - xác suất của lớp 1)
p_train_raw = best_model.predict_proba(X_train)[:, 1]
p_val_raw = best_model.predict_proba(X_val)[:, 1]
p_test_raw = best_model.predict_proba(X_test)[:, 1]
p_oot_raw = best_model.predict_proba(X_oot)[:, 1]

# Đánh giá trên tập kiểm định
val_auc_raw = roc_auc_score(y_val, p_val_raw)
val_brier_raw = brier_score_loss(y_val, p_val_raw)

# Đánh giá trên tập kiểm tra
test_auc_raw = roc_auc_score(y_test, p_test_raw)
test_brier_raw = brier_score_loss(y_test, p_test_raw)

print(f"\nKiểm định (chưa hiệu chỉnh):")
print(f"   AUC:   {val_auc_raw:.4f}")
print(f"   Brier: {val_brier_raw:.4f}")

print(f"\nKiểm tra (chưa hiệu chỉnh):")
print(f"   AUC:   {test_auc_raw:.4f}")
print(f"   Brier: {test_brier_raw:.4f}")


DỰ ĐOÁN THÔ (Chưa Hiệu Chỉnh)

Kiểm định (chưa hiệu chỉnh):
   AUC:   0.5612
   Brier: 0.0980

Kiểm tra (chưa hiệu chỉnh):
   AUC:   0.5354
   Brier: 0.0895


## 5. Hiệu Chỉnh Xác Suất

### 5.1. Isotonic Regression (Phương Pháp Chính)

In [8]:
# ============================================
# HIỆU CHỈNH BẰNG ISOTONIC REGRESSION
# ============================================

print("\n" + "=" * 70)
print("HIỆU CHỈNH BẰNG ISOTONIC REGRESSION")
print("=" * 70)

print(f"\nHuấn luyện trên TOÀN BỘ tập kiểm định: {len(y_val):,} mẫu")
print("   (Tập kiểm định KHÔNG BAO GIỜ được dùng để điều chỉnh)")

# Khởi tạo bộ hiệu chỉnh
isotonic = IsotonicRegression(
    y_min=0.001,            # Tránh giá trị đúng bằng 0
    y_max=0.999,            # Tránh giá trị đúng bằng 1
    out_of_bounds='clip'    # Cắt các giá trị ngoài khoảng
)

# Huấn luyện trên dự đoán của tập kiểm định
isotonic.fit(p_val_raw, y_val)

# Áp dụng hiệu chỉnh
p_val_iso = isotonic.predict(p_val_raw)
p_test_iso = isotonic.predict(p_test_raw)
p_oot_iso = isotonic.predict(p_oot_raw)

print("\nHiệu chỉnh Isotonic hoàn tất!")


HIỆU CHỈNH BẰNG ISOTONIC REGRESSION

Huấn luyện trên TOÀN BỘ tập kiểm định: 1,000 mẫu
   (Tập kiểm định KHÔNG BAO GIỜ được dùng để điều chỉnh)

Hiệu chỉnh Isotonic hoàn tất!


### 5.2. Platt Scaling (Phân Tích Độ Nhạy)

In [9]:
# ============================================
# PLATT SCALING (HỒI QUY LOGISTIC)
# ============================================

print("\n" + "=" * 70)
print("PLATT SCALING (Phân Tích Độ Nhạy)")
print("=" * 70)

print(f"\nHuấn luyện trên TOÀN BỘ tập kiểm định: {len(y_val):,} mẫu")

# Khởi tạo bộ Platt scaler (Hồi quy Logistic)
platt = LogisticRegression()

# Huấn luyện trên dự đoán của tập kiểm định
platt.fit(p_val_raw.reshape(-1, 1), y_val)

# Áp dụng hiệu chỉnh
p_val_platt = platt.predict_proba(p_val_raw.reshape(-1, 1))[:, 1]
p_test_platt = platt.predict_proba(p_test_raw.reshape(-1, 1))[:, 1]
p_oot_platt = platt.predict_proba(p_oot_raw.reshape(-1, 1))[:, 1]

print("\nPlatt Scaling hoàn tất!")


PLATT SCALING (Phân Tích Độ Nhạy)

Huấn luyện trên TOÀN BỘ tập kiểm định: 1,000 mẫu

Platt Scaling hoàn tất!


## 6. Expected Calibration Error (ECE)

In [10]:
# ============================================
# TÍNH EXPECTED CALIBRATION ERROR (ECE)
# ============================================

def expected_calibration_error(y_true, y_pred, bins=10):
    """
    Tính Expected Calibration Error.
    
    ECE đo lường độ lệch trung bình giữa độ chính xác thực tế
    và độ tin cậy dự đoán trong từng bin xác suất.
    
    Công thức:
    ECE = Σ (|độ_chính_xác_bin - độ_tin_cậy_bin|) × tỷ_lệ_bin
    
    Tham số:
        y_true: Nhãn thực tế (0 hoặc 1)
        y_pred: Xác suất dự đoán
        bins: Số lượng bins để chia (mặc định 10)
    
    Trả về:
        Giá trị ECE (càng thấp càng tốt)
    """
    # Tạo các cạnh bins đều nhau từ 0 đến 1
    bin_edges = np.linspace(0, 1, bins + 1)
    # Gán mỗi dự đoán vào một bin
    bin_ids = np.digitize(y_pred, bin_edges[1:-1])
    
    ece = 0.0
    # Tính ECE cho từng bin
    for b in range(bins):
        mask = (bin_ids == b)
        if mask.sum() > 0:
            accuracy = y_true[mask].mean()       # Độ chính xác thực tế trong bin
            confidence = y_pred[mask].mean()     # Độ tin cậy trung bình trong bin
            bin_proportion = mask.mean()         # Tỷ lệ mẫu trong bin
            ece += bin_proportion * abs(accuracy - confidence)
    
    return ece

print("Đã định nghĩa hàm ECE")

Đã định nghĩa hàm ECE


## 7. So Sánh Các Phương Pháp Hiệu Chỉnh

In [11]:
# ============================================
# SO SÁNH CÁC PHƯƠNG PHÁP HIỆU CHỈNH
# ============================================

print("\n" + "=" * 70)
print("SO SÁNH CÁC PHƯƠNG PHÁP HIỆU CHỈNH")
print("=" * 70)

# Tính toán các chỉ số đánh giá
results = pd.DataFrame({
    'Phương pháp': ['Thô (Raw)', 'Isotonic', 'Platt'],
    
    # Các chỉ số trên tập kiểm định
    'Val_AUC': [
        roc_auc_score(y_val, p_val_raw),
        roc_auc_score(y_val, p_val_iso),
        roc_auc_score(y_val, p_val_platt)
    ],
    'Val_Brier': [
        brier_score_loss(y_val, p_val_raw),
        brier_score_loss(y_val, p_val_iso),
        brier_score_loss(y_val, p_val_platt)
    ],
    'Val_ECE': [
        expected_calibration_error(y_val, p_val_raw),
        expected_calibration_error(y_val, p_val_iso),
        expected_calibration_error(y_val, p_val_platt)
    ],
    
    # Các chỉ số trên tập kiểm tra
    'Test_AUC': [
        roc_auc_score(y_test, p_test_raw),
        roc_auc_score(y_test, p_test_iso),
        roc_auc_score(y_test, p_test_platt)
    ],
    'Test_Brier': [
        brier_score_loss(y_test, p_test_raw),
        brier_score_loss(y_test, p_test_iso),
        brier_score_loss(y_test, p_test_platt)
    ],
    'Test_ECE': [
        expected_calibration_error(y_test, p_test_raw),
        expected_calibration_error(y_test, p_test_iso),
        expected_calibration_error(y_test, p_test_platt)
    ]
})

print("\nKẾT QUẢ SO SÁNH:")
print(results.round(4).to_string(index=False))

# Tính toán mức cải thiện
print("\nMỨC CẢI THIỆN (so với dự đoán thô):")
print("\nIsotonic Regression:")
print(f"   Val Brier:  {(results.loc[0, 'Val_Brier'] - results.loc[1, 'Val_Brier']) / results.loc[0, 'Val_Brier'] * 100:+.1f}%")
print(f"   Val ECE:    {(results.loc[0, 'Val_ECE'] - results.loc[1, 'Val_ECE']) / results.loc[0, 'Val_ECE'] * 100:+.1f}%")
print(f"   Test Brier: {(results.loc[0, 'Test_Brier'] - results.loc[1, 'Test_Brier']) / results.loc[0, 'Test_Brier'] * 100:+.1f}%")
print(f"   Test ECE:   {(results.loc[0, 'Test_ECE'] - results.loc[1, 'Test_ECE']) / results.loc[0, 'Test_ECE'] * 100:+.1f}%")

print("\nPlatt Scaling:")
print(f"   Val Brier:  {(results.loc[0, 'Val_Brier'] - results.loc[2, 'Val_Brier']) / results.loc[0, 'Val_Brier'] * 100:+.1f}%")
print(f"   Val ECE:    {(results.loc[0, 'Val_ECE'] - results.loc[2, 'Val_ECE']) / results.loc[0, 'Val_ECE'] * 100:+.1f}%")
print(f"   Test Brier: {(results.loc[0, 'Test_Brier'] - results.loc[2, 'Test_Brier']) / results.loc[0, 'Test_Brier'] * 100:+.1f}%")
print(f"   Test ECE:   {(results.loc[0, 'Test_ECE'] - results.loc[2, 'Test_ECE']) / results.loc[0, 'Test_ECE'] * 100:+.1f}%")


SO SÁNH CÁC PHƯƠNG PHÁP HIỆU CHỈNH

KẾT QUẢ SO SÁNH:
Phương pháp  Val_AUC  Val_Brier  Val_ECE  Test_AUC  Test_Brier  Test_ECE
  Thô (Raw)   0.5612     0.0980   0.0350    0.5354      0.0895    0.0529
   Isotonic   0.5949     0.0944   0.0000    0.5239      0.0884    0.0180
      Platt   0.5612     0.0961   0.0014    0.5354      0.0868    0.0126

MỨC CẢI THIỆN (so với dự đoán thô):

Isotonic Regression:
   Val Brier:  +3.8%
   Val ECE:    +99.9%
   Test Brier: +1.2%
   Test ECE:   +65.9%

Platt Scaling:
   Val Brier:  +2.0%
   Val ECE:    +96.0%
   Test Brier: +3.0%
   Test ECE:   +76.2%


## 8. Lựa Chọn Phương Pháp Chính

In [12]:
# ============================================
# LỰA CHỌN PHƯƠNG PHÁP HIỆU CHỈNH CHÍNH
# ============================================

print("\n" + "=" * 70)
print("LỰA CHỌN PHƯƠNG PHÁP CHÍNH")
print("=" * 70)

# Chọn Isotonic làm phương pháp chính (thường tốt hơn với đủ dữ liệu)
p_test_calibrated = p_test_iso
p_oot_calibrated = p_oot_iso

print("\nPHƯƠNG PHÁP CHÍNH: Isotonic Regression")
print("   Lý do: ECE cải thiện tốt hơn trên tập kiểm định")
print("\nPHÂN TÍCH ĐỘ NHẠY: Platt Scaling cũng được báo cáo")
print("   Kết quả tương tự → Xác nhận tính vững")

# Lưu các chỉ số cuối cùng
final_test_auc = roc_auc_score(y_test, p_test_calibrated)
final_test_brier = brier_score_loss(y_test, p_test_calibrated)
final_test_ece = expected_calibration_error(y_test, p_test_calibrated)

print(f"\nCÁC CHỈ SỐ KIỂM TRA CUỐI CÙNG (Hiệu chỉnh Isotonic):")
print(f"   AUC:   {final_test_auc:.4f}")
print(f"   Brier: {final_test_brier:.4f}")
print(f"   ECE:   {final_test_ece:.4f}")


LỰA CHỌN PHƯƠNG PHÁP CHÍNH

PHƯƠNG PHÁP CHÍNH: Isotonic Regression
   Lý do: ECE cải thiện tốt hơn trên tập kiểm định

PHÂN TÍCH ĐỘ NHẠY: Platt Scaling cũng được báo cáo
   Kết quả tương tự → Xác nhận tính vững

CÁC CHỈ SỐ KIỂM TRA CUỐI CÙNG (Hiệu chỉnh Isotonic):
   AUC:   0.5239
   Brier: 0.0884
   ECE:   0.0180


## 9. Xuất Mô Hình và Dự Đoán

In [13]:
# ============================================
# XUẤT MÔ HÌNH VÀ DỰ ĐOÁN
# ============================================

import pickle

# Thư mục đầu ra
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Đóng gói mô hình
model_package = {
    'base_model': best_model,                   # Mô hình cơ sở đã được điều chỉnh
    'isotonic_calibrator': isotonic,            # Bộ hiệu chỉnh Isotonic
    'platt_calibrator': platt,                  # Bộ hiệu chỉnh Platt
    'best_params': cv_model.best_params_,       # Siêu tham số tốt nhất
    'modeling_features': MODELING_FEATURES,     # Danh sách đặc trưng
    'cat_indices': cat_indices,                 # Chỉ số đặc trưng phân loại
    'seed': SEED                                # Seed để tái lập
}

# Lưu mô hình ra file pkl
model_path = output_dir / "predictive_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model_package, f)

print(f"Đã lưu mô hình: {model_path}")

# Tạo DataFrame chứa các dự đoán
predictions_df = pd.DataFrame({
    'user_id': df['user_id'],
    'split': df['split'],
    'y_true': df['conversion'],
    'p_raw': np.concatenate([p_train_raw, p_val_raw, p_test_raw, p_oot_raw]),
    'p_isotonic': np.concatenate([
        isotonic.predict(p_train_raw),
        p_val_iso,
        p_test_iso,
        p_oot_iso
    ]),
    'p_platt': np.concatenate([
        platt.predict_proba(p_train_raw.reshape(-1, 1))[:, 1],
        p_val_platt,
        p_test_platt,
        p_oot_platt
    ])
})

# Lưu dự đoán ra file parquet
predictions_path = output_dir / "predictions_calibrated.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print(f"Đã lưu dự đoán: {predictions_path}")

# Lưu các chỉ số đánh giá
metrics = {
    'test_auc': float(final_test_auc),
    'test_brier': float(final_test_brier),
    'test_ece': float(final_test_ece),
    'comparison_table': results.to_dict('records')
}

metrics_path = output_dir / "predictive_metrics.json"
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Đã lưu các chỉ số: {metrics_path}")

Đã lưu mô hình: ../data/processed/predictive_model.pkl
Đã lưu dự đoán: ../data/processed/predictions_calibrated.parquet
Đã lưu các chỉ số: ../data/processed/predictive_metrics.json


## 10. Tóm Tắt

In [14]:
# ============================================
# TÓM TẮT CUỐI CÙNG
# ============================================

print("\n" + "=" * 70)
print("TÓM TẮT NOTEBOOK 02")
print("=" * 70)

print(f"\nĐiều chỉnh siêu tham số: 5-fold CV trên {len(X_train):,} mẫu huấn luyện")
print(f"   Tham số tốt nhất: {cv_model.best_params_}")

print(f"\nHiệu chỉnh: TOÀN BỘ tập kiểm định ({len(X_val):,} mẫu)")
print(f"   Isotonic Regression (chính)")
print(f"   Platt Scaling (độ nhạy)")

print(f"\nCác chỉ số kiểm tra cuối cùng (Isotonic):")
print(f"   AUC:   {final_test_auc:.4f}")
print(f"   Brier: {final_test_brier:.4f}")
print(f"   ECE:   {final_test_ece:.4f}")

print("\nCÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:")
print("   - Điều chỉnh bằng CV trên tập huấn luyện (KHÔNG dùng tập kiểm định)")
print("   - Hiệu chỉnh trên TOÀN BỘ 1,000 mẫu kiểm định")
print("   - So sánh: Isotonic vs. Platt")
print("   - Chỉ dùng đặc trưng pre-treatment (không rò rỉ)")

print("\nSẵn sàng cho Notebook 03 (Ước Lượng CATE)")
print("=" * 70)


TÓM TẮT NOTEBOOK 02

Điều chỉnh siêu tham số: 5-fold CV trên 3,000 mẫu huấn luyện
   Tham số tốt nhất: {'learning_rate': 0.1, 'max_depth': 4, 'max_iter': 50}

Hiệu chỉnh: TOÀN BỘ tập kiểm định (1,000 mẫu)
   Isotonic Regression (chính)
   Platt Scaling (độ nhạy)

Các chỉ số kiểm tra cuối cùng (Isotonic):
   AUC:   0.5239
   Brier: 0.0884
   ECE:   0.0180

CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:
   - Điều chỉnh bằng CV trên tập huấn luyện (KHÔNG dùng tập kiểm định)
   - Hiệu chỉnh trên TOÀN BỘ 1,000 mẫu kiểm định
   - So sánh: Isotonic vs. Platt
   - Chỉ dùng đặc trưng pre-treatment (không rò rỉ)

Sẵn sàng cho Notebook 03 (Ước Lượng CATE)
